# Capstone — Content-Day Performance and Decline-Risk Triage

**Lane:** Refresh / Content Opportunity Scoring.  
**Research question:** Using signals available by March 15, 2026, can we rank eligible content items by their likelihood of showing a meaningful decline in GSC clicks per measured day during March 16–31, so reviewers can prioritize investigation?

This notebook consolidates the Weeks 4–7 work into a reproducible capstone record. It does not retrain the models or invent missing outputs. Run the weekly notebooks first, then run this notebook. The public paper is `docs/index.html` and must be updated only with verified outputs.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'work').exists():
    ROOT = Path('/content/FlyRank_Internship')
OUT = ROOT / 'work' / 'outputs'
FIG = ROOT / 'work' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

def load_json(name, required=False):
    path = OUT / name
    if not path.exists():
        if required:
            raise FileNotFoundError(f'Missing {path}. Run the relevant weekly notebook and upload its receipt.')
        print(f'Optional receipt not found: {name}')
        return None
    return json.loads(path.read_text())

w05 = load_json('w05_metrics.json', required=True)
w07 = load_json('w07_playbook_metrics.json')
w04 = load_json('baseline_metrics.json')
print('Loaded Week 5 metrics. Week 4/7 receipts:', bool(w04), bool(w07))


## 1. Data and study design

**Source:** FlyRank ML Internship pseudonymized warehouse release, build `flyrank_pseudonymized_warehouse_release_v20260703`. The analysis uses `fact_content_daily_performance` for daily Google Search Console observations and `dim_content` for content dates. The fact grain is report date × client × content.

**Windows:** March 1–15, 2026 is the feature window; March 16–31, 2026 is the outcome window; prediction cutoff is March 16. Rows are treated as measured only when `gsc_data_available IS TRUE`. Eligibility requires at least 10 measured days in each half, at least 100 early impressions, and at least 5 early clicks. The June sample partition is excluded from model development. Pseudonymous identifiers are for grouping and local queue management only and must not appear in the public paper.


## 2. Methodology

**Outcome proxy:** a page is labeled 1 when clicks per measured day in March 16–31 are at least 20% lower than in March 1–15; otherwise 0. This is an observed change proxy, not a human-verified content-quality label.

**Features:** early impressions, early clicks, early clicks per day, early average position, early measured days, content age, and days since update. Client identifiers are grouping variables, not model features.

**Models:** regularized Logistic Regression and Random Forest. **Validation:** a client-group holdout (seed 42; 20% test) is the headline evaluation; random-row split is diagnostic only and is not reported if its receipt is absent. **Metrics:** base rate, Precision@10/20/50, Average Precision and ROC-AUC.

**Leakage controls:** later-window metrics and target-derived fields are excluded from features. Missing GSC observations are not silently treated as measured zeroes.


In [ ]:
comparison = pd.DataFrame(w05['comparison'])
display(comparison)
print('Grouped-client test base rate:', w05.get('test_base_rate', 'not present in receipt'))
print('Random-row diagnostic:', w05.get('random_split_logistic_diagnostic', 'not saved; omitted'))
if w07:
    print('Week 7 selected method:', w07.get('selected_method', 'not present'))
    print('Week 7 held-out pages:', w07.get('heldout_pages', 'not present'))


## 3. Important baseline comparability note

The frozen Week-4 baseline was documented as `log1p(early_impressions)`. The Week-5 notebook's comparison code applies a freshness multiplier to that score. Therefore, the Week-5 baseline row is not an exact reproduction of the frozen Week-4 score. Report the Week-5 comparison as the comparison actually executed, disclose this implementation difference, and do not describe it as a clean head-to-head against the exact frozen Week-4 rule. A strict apples-to-apples rerun would be needed to resolve this discrepancy.


## 4. Interpretation and limitations

The Week-5 metrics supplied for this study show Logistic Regression with Precision@20 of 0.70, compared with 0.50 for the Week-5 baseline implementation; Random Forest has Precision@20 of 0.70. These are results for the recorded grouped-client test split, not proof of future-month performance. The base rate is 0.542363. The Week-5 baseline implementation differs from the frozen Week-4 rule as described above.

The label can reflect query-mix changes, seasonality, SERP layout, ranking shifts, measurement coverage, or tracking changes. The evaluation covers one March window and does not include a multi-month temporal test. No controlled refresh intervention was evaluated, so the analysis cannot claim that refreshing content causes clicks to recover. Treat scores as decision support for human investigation, not automatic content changes.


## 5. Ranked recommendations and human review

The Week-7 action playbook ranks held-out items using Logistic Regression when its Precision@20 is at least the Week-5 baseline implementation's Precision@20; otherwise it retains the baseline. It assigns `priority_review`, `review`, and `monitor` tiers, with operational archetypes `priority_decline_review`, `model_rule_disagreement`, and `monitor`.

For high-priority items, first verify GSC coverage and tracking, then inspect query mix, seasonality, SERP changes, position and page intent. Only after these checks should a human reviewer consider content edits. Do not publish page identifiers, client labels, raw URLs or private queries.


## 6. Reproducibility and data credit

Repository: https://github.com/yaseen20051/FlyRank_Internship  
Weekly notebooks: `work/notebooks/w04_baseline_score.ipynb`, `w05_model.ipynb`, `w06_validation_audit.ipynb`, `w07_action_playbook.ipynb`.  
Data credit: **Built on the FlyRank ML Internship dataset** — https://flyrank.ai.
